# ML Assignment 02

---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [1]:
# Question 1
import pandas as pd

df = pd.read_csv("house_price_regression_dataset.csv")

print(f"-----------Dataset Shape----------")
print(df.shape)

print(f"\n-----------First 10 rows----------")
display(df.head(10))

print(f"\n-----------5 random samples----------")
display(df.sample(5))

-----------Dataset Shape----------
(1000, 8)

-----------First 10 rows----------


,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06
5,3944,5,3,1990,2.475930,2,8,8.797970e+05
6,3671,1,2,2012,4.911960,0,1,8.144279e+05
7,3419,1,1,1972,2.805281,1,1,7.034131e+05
8,630,3,3,1997,1.014286,1,8,1.738750e+05
9,2185,4,2,1981,3.941604,2,5,5.041765e+05



-----------5 random samples----------


,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
237,1545,1,2,1970,3.173373,1,3,338170.315281
42,3112,2,3,1980,2.656670,0,9,666733.473462
702,2862,1,2,1968,4.767132,0,2,613658.109947
572,904,3,3,1957,1.212368,0,5,198204.128305
395,3046,5,2,1956,3.661198,0,5,676719.122059


---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [2]:
#Question 2
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

thresh = len(df) * 0.5
df_cleaned = df.dropna(thresh=thresh, axis=1)

X = df_cleaned.drop(columns=["House_Price"])
y = df_cleaned["House_Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns


num_imputer = SimpleImputer(strategy="mean")
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
    X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

print("--- Final Dataset Shapes ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

--- Final Dataset Shapes ---
X_train shape: (800, 7)
X_test shape:  (200, 7)
y_train shape: (800,)
y_test shape:  (200,)


---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [3]:
import numpy as np

X_train_raw = np.array(X_train["Square_Footage"])
X_test_raw = np.array(X_test["Square_Footage"])

y_train = np.array(y_train)
y_test = np.array(y_test)

X_min = X_train_raw.min()
X_max = X_train_raw.max()

X_train_scaled = (X_train_raw - X_min) / (X_max - X_min)
X_test_scaled = (X_test_raw - X_min) / (X_max - X_min)

lr = 0.1
epochs = 1000
n = float(len(y_train))

m = 0.0
c = 0.0

for epoch in range(epochs):
    y_pred = (m * X_train_scaled) + c
    error = y_pred - y_train

    dm = (2 / n) * np.sum(X_train_scaled * error)
    dc = (2 / n) * np.sum(error)

    m = m - (lr * dm)
    c = c - (lr * dc)

y_pred_test = (m * X_test_scaled) + c

print("--- Gradient Descent Results ---")
print(f"Learned Slope (m): {m:.4f}")
print(f"Learned Intercept (c): {c:.4f}")
print("\nFirst 5 Test Predictions vs Actual:")
for i in range(5):
    print(f"Predicted: {y_pred_test[i]:.2f} | Actual: {y_test[i]:.2f}")

--- Gradient Descent Results ---
Learned Slope (m): 901700.4877
Learned Intercept (c): 155110.9352

First 5 Test Predictions vs Actual:
Predicted: 858862.49 | Actual: 901000.49
Predicted: 517515.91 | Actual: 494537.51
Predicted: 998449.58 | Actual: 949404.20
Predicted: 1043374.16 | Actual: 1040389.05
Predicted: 785458.94 | Actual: 794010.02


---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [4]:
#Question 4
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"]
cat_cols = ["Neighborhood_Quality"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="passthrough",
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("--- ColumnTransformer Verification ---")
print(f"Original X_train shape:    {X_train.shape}")
print(f"Transformed X_train shape: {X_train_transformed.shape}")

--- ColumnTransformer Verification ---
Original X_train shape:    (800, 7)
Transformed X_train shape: (800, 16)


## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [5]:
# Question 5
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

sgd_pipe = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", SGDRegressor(random_state=42, max_iter=1000))
    ]
)

sgd_pipe.fit(X_train, y_train)
y_pred = sgd_pipe.predict(X_test)

print("--- Pipeline Evaluation Metrics ---")
print(f"RMSE: {round(root_mean_squared_error(y_test, y_pred), 4)}")
print(f"R2 score: {round(r2_score(y_test, y_pred), 4)}")

print("\n--- First 10 Test Samples: Predicted vs Actual ---")
for i in range(10):
    print(f"Sample {i+1}: Predicted: {y_pred[i]:.2f} | Actual: {y_test[i]:.2f}")

--- Pipeline Evaluation Metrics ---
RMSE: 29096.4609
R2 score: 0.9869

--- First 10 Test Samples: Predicted vs Actual ---
Sample 1: Predicted: 851369.91 | Actual: 901000.49
Sample 2: Predicted: 507342.82 | Actual: 494537.51
Sample 3: Predicted: 985831.68 | Actual: 949404.20
Sample 4: Predicted: 1022464.74 | Actual: 1040389.05
Sample 5: Predicted: 759809.08 | Actual: 794010.02
Sample 6: Predicted: 765675.68 | Actual: 724033.56
Sample 7: Predicted: 1004527.78 | Actual: 998439.24
Sample 8: Predicted: 903546.05 | Actual: 909713.44
Sample 9: Predicted: 797193.08 | Actual: 792681.52
Sample 10: Predicted: 888626.74 | Actual: 947490.78


---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [6]:
# Question 6
from sklearn.linear_model import LinearRegression

lr_pipe = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", LinearRegression()),
    ]
)

lr_pipe.fit(X_train, y_train)
y_pred = lr_pipe.predict(X_test)

print("--- Multiple Linear Regression Metrics ---")
print(f"RMSE: {round(root_mean_squared_error(y_test, y_pred), 4)}")
print(f"R2 score: {round(r2_score(y_test, y_pred), 4)}")

print("\n--- First 10 Test Samples: Predicted vs Actual ---")
for i in range(10):
    print(
        f"Sample {i+1}: Predicted: {y_pred[i]:.2f} | Actual: {y_test[i]:.2f}"
    )

--- Multiple Linear Regression Metrics ---
RMSE: 29093.0752
R2 score: 0.9869

--- First 10 Test Samples: Predicted vs Actual ---
Sample 1: Predicted: 851409.15 | Actual: 901000.49
Sample 2: Predicted: 507641.03 | Actual: 494537.51
Sample 3: Predicted: 985990.38 | Actual: 949404.20
Sample 4: Predicted: 1022654.71 | Actual: 1040389.05
Sample 5: Predicted: 760698.13 | Actual: 794010.02
Sample 6: Predicted: 765782.32 | Actual: 724033.56
Sample 7: Predicted: 1005039.10 | Actual: 998439.24
Sample 8: Predicted: 903407.78 | Actual: 909713.44
Sample 9: Predicted: 797971.19 | Actual: 792681.52
Sample 10: Predicted: 888850.43 | Actual: 947490.78


---
## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [7]:
# Question 7
df_sample = df.sample(n=100, random_state=42)

X_raw = df_sample[
    ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"]
].values
y = df_sample["House_Price"].values

X_b = np.c_[np.ones((len(X_raw), 1)), X_raw]

theta = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)

print("--- Learned Coefficients (Theta Values) ---")
print(f"Intercept (theta_0): {theta[0]:.4f}")
print(f"Square_Footage Coefficient (theta_1): {theta[1]:.4f}")
print(f"Num_Bedrooms Coefficient (theta_2): {theta[2]:.4f}")
print(f"Num_Bathrooms Coefficient (theta_3): {theta[3]:.4f}")

--- Learned Coefficients (Theta Values) ---
Intercept (theta_0): 20888.2595
Square_Footage Coefficient (theta_1): 202.3082
Num_Bedrooms Coefficient (theta_2): 8303.2810
Num_Bathrooms Coefficient (theta_3): 3020.3285
